In [1]:
:set -XRankNTypes
:set -XExistentialQuantification
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XScopedTypeVariables
import Data.List (intercalate)
import Data.Maybe (fromMaybe)
putStrLn "Setup done." 

Setup done.

# 🔭 Расширения Кана в Haskell

Расширения Кана — наиболее общие конструкции в теории категорий.
МакЛейн: *«Все концепции — это расширения Кана».*

## Содержание
1. Правое расширение Кана (Ran)
2. Левое расширение Кана (Lan)
3. Конкретные примеры: Ran/Lan для Maybe, [], Either
4. Монада плотности (Codensity)
5. Комонада плотности (Density)
6. Лемма Ёнеды как частный случай Ran
7. Сопряжения через Ran
8. Монада из сопряжения
9. Оптимизация через Codensity
---

## 1️⃣ Правое расширение Кана (Ran)

Для функторов `g : C -> D` и `h : C -> E` правое расширение Кана
`Ran g h : D -> E` — это «наилучшее приближение» к продолжению `h` вдоль `g`.

В Haskell (с RankNTypes):

```
newtype Ran g h a = Ran { runRan :: forall b. (a -> g b) -> h b }
```

Это кодирует: для всех `b`, при наличии `a -> g b`, получаем `h b`.

In [2]:
-- Ran g h a = forall b. (a -> g b) -> h b
newtype Ran g h a = Ran { runRan :: forall b. (a -> g b) -> h b }

instance Functor (Ran g h) where
  fmap f (Ran r) = Ran (\k -> r (k . f))

-- fromRan: direct constructor
fromRan :: (forall b. (a -> g b) -> h b) -> Ran g h a
fromRan f = Ran f

putStrLn "Определения Ran OK."

Line 9: Eta reduce
Found:
fromRan f = Ran f
Why not:
fromRan = Ran

Определения Ran OK.

### Диаграмма: Правое расширение Кана

![Ran: универсальное свойство](../diagrams/kan/ran_diagram.svg)

`Ran g h` — "лучшее приближение" `h` вдоль `g`

## 2️⃣ Левое расширение Кана (Lan)

Левое расширение Кана `Lan g h : D -> E` двойственно Ran.

В Haskell (экзистенциальное кодирование):

```
data Lan g h a = forall b. Lan (g b -> a) (h b)
```

Это кодирует: существует `b`, с функцией `g b -> a` и значением `h b`.

In [3]:
-- Left Kan extension: Lan g h a = exists b. (g b -> a, h b)
-- Represented as a pair: (Maybe Int -> Int, [Int])
-- This simulates "Lan Maybe [] Int"

-- A Lan [] Maybe Int value = (f :: [Int] -> Int, xs :: [Int])
-- where f uses the list to compute an Int
-- Extract: apply f to xs (via head or 0 default)
type LanPair = (Maybe Int -> Int, [Int])

mkLanPair :: [Int] -> LanPair
mkLanPair xs = (\mb -> case mb of { Nothing -> 0; Just x -> x }, xs)

extractLanPair :: LanPair -> Int
extractLanPair (f, xs) = f (if null xs then Nothing else Just (head xs))

fmapLanPair :: (Int -> Int) -> LanPair -> LanPair
fmapLanPair g (f, xs) = (g . f, xs)

putStrLn "Lan g h a = exists b. (g b -> a, h b)"

let lan1 = mkLanPair [10, 20, 30]
let lan2 = mkLanPair []
putStrLn $ "extractLanPair [10,20,30] = " ++ show (extractLanPair lan1)
putStrLn $ "extractLanPair []         = " ++ show (extractLanPair lan2)

let lan3 = fmapLanPair (+100) lan1
putStrLn $ "fmapLanPair (+100) [10,20,30] = " ++ show (extractLanPair lan3)
putStrLn "Lan demo complete." 

Lan g h a = exists b. (g b -> a, h b)

extractLanPair [10,20,30] = 10

extractLanPair []         = 0

fmapLanPair (+100) [10,20,30] = 110

Lan demo complete.

### Диаграмма: Левое расширение Кана

![Lan: универсальное свойство](../diagrams/kan/lan_diagram.svg)

`Lan g h` — «свободное продолжение» `h` вдоль `g`

## 3️⃣ Конкретные примеры: Ran/Lan для Maybe, [], Either

Расширения Кана позволяют **выразить любой функтор** через универсальное свойство.

- **`Ran Just Id ≅ Maybe`** — правый сопряжённый для `Just`
- **`Lan (:[]) Id ≅ []`** — левый сопряжённый для `singleton`
- **`Ran Left Id ≅ Either e`** — правый сопряжённый для `Left`

### Диаграмма

![Concrete examples: Ran/Lan для Maybe, [], Either](kan_examples.svg)

In [4]:
-- Ran Maybe Maybe ≅ Maybe: round-trip
-- fromRanMaybe recovers Maybe
fromRanMaybe :: Ran Maybe Maybe a -> Maybe a
fromRanMaybe r = runRan r Just

-- toRanMaybe lifts Maybe
toRanMaybe :: Maybe a -> Ran Maybe Maybe a
toRanMaybe ma = Ran (\k -> ma >>= k)

let t1 = fromRanMaybe (toRanMaybe (Just 42 :: Maybe Int))
let t2 = fromRanMaybe (toRanMaybe (Nothing :: Maybe Int))
print t1
print t2

Just 42

Nothing

In [5]:
-- Левое Kan через existential type
data LanList a = forall b. LanList ([b] -> a) [b]

-- fromLanList: кодируем [a] как Lan [] Id a
fromLanList :: [a] -> LanList a
fromLanList xs = LanList (\(y:_) -> y) xs

-- toLanList: извлекаем список
toLanList :: LanList a -> [a]
toLanList (LanList f bs) = map (f . (:[])) bs

-- fmapLanL
fmapLanL :: (a -> c) -> LanList a -> LanList c
fmapLanL g (LanList f bs) = LanList (g . f) bs

let ll = fromLanList [1,2,3 :: Int]
let ll2 = fmapLanL (*2) ll
print (toLanList ll)
print (toLanList ll2)

[1,2,3]

[2,4,6]

## 4️⃣ Монада плотности (Codensity)

**Монада плотности** функтора `f` — это `Ran f f`. Любой функтор порождает монаду. CPS-вычисления организуют `>>=` правоассоциативно.

### Диаграмма

![Codensity: CPS-преобразование и оптимизация](../diagrams/kan/kan_codensity.svg)

In [6]:
-- Codensity = Ran f f
newtype Codensity f a = Codensity { runCodensity :: forall b. (a -> f b) -> f b }
instance Functor (Codensity f) where
  fmap f (Codensity c) = Codensity (\k -> c (k . f))
instance Applicative (Codensity f) where
  pure a = Codensity (\k -> k a)
  (Codensity cf) <*> (Codensity ca) = Codensity (\k -> cf (\f -> ca (k . f)))
instance Monad (Codensity f) where
  return = pure
  (Codensity c) >>= f = Codensity (\k -> c (\a -> runCodensity (f a) k))
liftCodensity :: Monad m => m a -> Codensity m a
liftCodensity m = Codensity (m >>=)
lowerCodensity :: Monad m => Codensity m a -> m a
lowerCodensity (Codensity c) = c return
putStrLn "Определения Codensity OK."

Определения Codensity OK.

## 5️⃣ Комонада плотности (Density)

**Комонада плотности** `f` — это `Lan f f`. Симметрия: `Codensity = Ran f f` (монада) vs `Density = Lan f f` (комонада).

### Диаграмма

![Density comonad: Lan f f](../diagrams/kan/kan_density.svg)

In [7]:
data Density f a = forall b. Density (f b -> a) (f b)
instance Functor (Density f) where
  fmap f (Density g fb) = Density (f . g) fb
extractD :: Density f a -> a
extractD (Density f fb) = f fb
duplicateD :: Density f a -> Density f (Density f a)
duplicateD (Density f fb) = Density (Density f) fb
newtype Id a = Id { unId :: a } deriving (Functor)
mkStore :: (s -> a) -> s -> Density Id a
mkStore f s = Density (f . unId) (Id s)
putStrLn "Комонада Density OK."

Комонада Density OK.

## 6️⃣ Лемма Ёнеды как частный случай Ran

При `g = Id`, `Ran Id h ≅ h`: `Nat(Hom(a,-), h) ≅ h a`.

### Диаграмма

![Yoneda as special case of Ran Id](../diagrams/kan/kan_yoneda.svg)

In [8]:
newtype Yoneda f a = Yoneda { runYoneda :: forall b. (a -> b) -> f b }
instance Functor (Yoneda f) where
  fmap f (Yoneda y) = Yoneda (\k -> y (k . f))
toYoneda :: Functor f => f a -> Yoneda f a
toYoneda fa = Yoneda (\f -> fmap f fa)
fromYoneda :: Yoneda f a -> f a
fromYoneda (Yoneda y) = y id
let xs = [1,2,3 :: Int]
print (fromYoneda (toYoneda xs))
let result = fromYoneda $ fmap (*3) $ fmap (+1) $ toYoneda [1..5 :: Int]
print result

[1,2,3]

[6,9,12,15,18]

## 7️⃣ Сопряжения через Ran

Если `f ⊣ g`, то `Ran f Id ≅ g`, `Lan g Id ≅ f`.

- Правый сопряжённый = правое Kan тождественного функтора вдоль `f`
- Левый сопряжённый = левое Kan тождественного функтора вдоль `g`

### Диаграмма

![Adjunctions via Kan extensions](../diagrams/kan/kan_adjunction.svg)

In [9]:
adjUnit :: a -> e -> (e, a)
adjUnit a e = (e, a)
adjCounit :: (e, e -> a) -> a
adjCounit (e, f) = f e
type MyState s a = s -> (a, s)
bind :: MyState s a -> (a -> MyState s b) -> MyState s b
bind m f = \s -> let (a, s') = m s in f a s'
return' :: a -> MyState s a
return' a = \s -> (a, s)
putStrLn "Сопряжения OK."

Line 7: Redundant lambda
Found:
bind m f = \ s -> let (a, s') = m s in f a s'
Why not:
bind m f s = let (a, s') = m s in f a s'Line 9: Redundant lambda
Found:
return' a = \ s -> (a, s)
Why not:
return' a s = (a, s)Line 9: Use tuple-section
Found:
\ s -> (a, s)
Why not:
(a,)

Сопряжения OK.

## 8️⃣ Монада из сопряжения

Любое сопряжение `f ⊣ g` порождает монаду `T = g · f`:

```
return = η;  join = g·ε·f
```

Примеры: `(e,) ⊣ (e->)` → `State`, `Free ⊣ Forget` → `[]`, `Maybe ⊣ Id` → `Maybe`.

In [10]:
newtype StateM s a = StateM { runStateM :: s -> (a, s) }
instance Functor (StateM s) where
  fmap f (StateM g) = StateM (\s -> let (a, s') = g s in (f a, s'))
instance Applicative (StateM s) where
  pure a = StateM (\s -> (a, s))
  StateM sf <*> StateM sa = StateM (\s ->
    let (f, s1) = sf s; (a, s2) = sa s1 in (f a, s2))
instance Monad (StateM s) where
  return = pure
  StateM sa >>= f = StateM (\s -> let (a, s') = sa s in runStateM (f a) s')
getM :: StateM s s
getM = StateM (\s -> (s, s))
putM :: s -> StateM s ()
putM s = StateM (\_ -> ((), s))
let result = runStateM (do { x <- getM; putM (x + 1); getM }) 10
print result

(11,11)

## 9️⃣ Оптимизация через Codensity

`Codensity` организует `>>=` правоассоциативно, улучшая O(n²) до O(n). **DList** — это `Codensity []`.

### Диаграмма

![Codensity: DList vs naive](../diagrams/kan/kan_codensity.svg)

In [11]:
newtype DList a = DList { runDList :: [a] -> [a] }
emptyDL :: DList a
emptyDL = DList id
singletonDL :: a -> DList a
singletonDL x = DList (x:)
appendDL :: DList a -> DList a -> DList a
appendDL (DList f) (DList g) = DList (f . g)
toListDL :: DList a -> [a]
toListDL (DList f) = f []
fromListDL :: [a] -> DList a
fromListDL xs = DList (xs++)
naiveConcat :: [[Int]] -> [Int]
naiveConcat = foldl (++) []
dlistConcat :: [[Int]] -> [Int]
dlistConcat xss = toListDL (foldl step emptyDL xss)
  where step acc xs = appendDL acc (fromListDL xs)
let r1 = naiveConcat [[1,2,3],[4,5,6]] :: [Int]
let r2 = dlistConcat [[1,2,3],[4,5,6]]
print (r1 == r2)
putStrLn "DList OK."

True

DList OK.

---
**← Предыдущий:** [Adjunctions](Adjunctions.ipynb)  |  **Следующий →** [MetaProgramming](MetaProgramming.ipynb)